In [ ]:
from google.colab import userdata

API_KEY = userdata.get("GROQ_API_KEY")

from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


In [ ]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
     response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
             {"role": "user",   "content": user_prompt},
         ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
     return response.choices[0].message.content

# TODO: Call it once with a simple question and print the answer.
answer = ask_llm("What is the capital of Ghana?")
print(answer)
# TODO: Print response.usage as well — how many tokens did your call consume?

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of Ghana?"},
    ],
    temperature=0.7,
    max_tokens=500,
)
print(response.choices[0].message.content)
print(response.usage)

The capital of Ghana is Accra.
The capital of Ghana is Accra.
CompletionUsage(completion_tokens=9, prompt_tokens=48, total_tokens=57, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.181553582, prompt_time=0.0022171, completion_time=0.013437265, total_time=0.015654365)


In [10]:
"""
The call consumed 57 tokens
Student Reasoning — Anatomy of a call
1. What is the difference between the system and user roles? Give an example of something that belongs in each.
The system role is an instruction about how the system should behave an example is "You are a helpful assistant".
While the user role is the actual task that is given to the system to perform an action an example is what is the capital of Ghana?

2. What is a token, roughly? Why do API providers bill per token rather than per request?
A token is a chunk of word, which is not the full word itself and not also a single character.  API providers bill per token because
compute cost scales with tokens, and not request."""

'\nThe call consumed 57 tokens \nStudent Reasoning — Anatomy of a call \n1. What is the difference between the system and user roles? Give an example of something that belongs in each.\nThe system role is an instruction about how the system should behave an example is "You are a helpful assistant". \nWhile the user role is the actual task that is given to the system to perform an action an example is what is the capital of Ghana?\n\n2. What is a token, roughly? Why do API providers bill per token rather than per request?\nA token is a chunk of word, which is not the full word itself and not also a single character.  API providers bill per token because \ncompute cost scales with tokens, and not request.'

In [ ]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.

question = "Suggest a name for a savings product for market traders in Accra."

print("Temperature = 0.0")
temp_0_answers = []
for i in range(5):
    answer = ask_llm(question, temperature=0.0)
    temp_0_answers.append(answer)
    print(f"\nRun {i+1}")
    print(answer)

print("\n\nTemperature = 1.2")
temp_12_answers = []
for i in range(5):
    answer = ask_llm(question, temperature=1.2)
    temp_12_answers.append(answer)
    print(f"\nRun {i+1}")
    print(answer)

Temperature = 0.0

Run 1
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth, which is a key goal for market traders.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "save" or "keep", so this name is simple and straightforward.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.
7. **Traders' Trust**: This name emphasizes the ide

In [2]:
"""Student Reasoning — Temperature What did you observe at each temperature?
For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?

At temperature 0.0 the mode always pick the highest probability next token, so theres no randomness to produce variation.
At temperature 1.2 higher temperature flattens the probability distribution so the model smaples lower probability tokens more often.
For the loan system the temperatue 0.0 is the appropriate choice because you need the same letter to produce the same structured facts
and the same risk assessment every time"""

'Student Reasoning — Temperature What did you observe at each temperature?\nFor the loan decision-support system you are about to build, which temperature regime is appropriate, and why?\n\nAt temperature 0.0 the mode always pick the highest probability next token, so theres no randomness to produce variation.\nAt temperature 1.2 higher temperature flattens the probability distribution so the model smaples lower probability tokens more often.\nFor the loan system the temperatue 0.0 is the appropriate choice because you need the same letter to produce the same structured facts \nand the same risk assessment every time'

In [ ]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")


6 letters loaded.


In [ ]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

SUMMARY_PROMPT_V1 = "Summarize this:"

def summarize_v1(letter_text):
    prompt = f"{SUMMARY_PROMPT_V1}\n\n{letter_text}"
    answer = ask_llm(prompt)
    return answer

print("V1 — L002")
print(summarize_v1(LETTERS["L002"]))

print("\nV1 — L006")
print(summarize_v1(LETTERS["L006"]))

SUMMARY_SYSTEM_PROMPT_V2 = (
    "You are an assistant to a microfinance loan officer. "
    "Summarize loan application letters factually and neutrally. "
    "Do not invent, assume, or infer any detail not explicitly stated in the letter. "
    "Keep the summary to 3-4 sentences."
)

def summarize_v2(letter_text):
    user_prompt = f"Summarize this loan application:\n\n{letter_text}"
    answer = ask_llm(
        user_prompt,
        system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
        temperature=0.0
    )
    return answer

print("V2 — L002")
print(summarize_v2(LETTERS["L002"]))

print("\nV2 — L006")
print(summarize_v2(LETTERS["L006"]))

print("L002 — V1 vs V2")

print("\nV1")
print(summarize_v1(LETTERS["L002"]))
print("\nV2")
print(summarize_v2(LETTERS["L002"]))

print("L006 — V1 vs V2")
print("\nV1")
print(summarize_v1(LETTERS["L006"]))
print("\nV2")
print(summarize_v2(LETTERS["L006"]))

V1 — L002
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow period in business, but expects it to improve after the festive season and is willing to repay the loan as soon as possible, despite not having collateral.

V1 — L006
Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. Although he has no experience and no collateral, he claims to be "business-minded" and promises to repay the loan within a year when his businesses are successful, relying on his personal trustworthiness.
V2 — L002
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but he expects it to improve after the festive season. He does not curr

In [3]:
"""Student Reasoning — Summarization prompts 1. What concrete problems did V1's output have that V2 fixed? Quote examples.
V1 has no length constraint, so its L002 summary might run to a full paragraph while its L006 summary is two sentences.
V1 sometimes adds evaluative language not asked for, e.g., turning "I need GHS 25,000 urgently... I do not have collateral"
into something like "Kwame appears to be a hardworking driver seeking support during a temporary rough patch," which imports a
sympathetic framing the letter itself does not establish. V2 fixed these by giving it a role and explicit constraints.

2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?
The instruction is essential beacuse hallucination here is compliance and fairnes risk, if the summariser envents details or fabricated
claim that is not stated in the letter it could influence a lending decision which might later poses danger to the loan officer.
The failure mode is called hallucination in the LLM literature, when a model generates content that sounds fluent and plausible but is
not grounded in the actual input or any verified fact."""

'Student Reasoning — Summarization prompts 1. What concrete problems did V1\'s output have that V2 fixed? Quote examples. \nV1 has no length constraint, so its L002 summary might run to a full paragraph while its L006 summary is two sentences. \nV1 sometimes adds evaluative language not asked for, e.g., turning "I need GHS 25,000 urgently... I do not have collateral" \ninto something like "Kwame appears to be a hardworking driver seeking support during a temporary rough patch," which imports a \nsympathetic framing the letter itself does not establish. V2 fixed these by giving it a role and explicit constraints.\n\n2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?\nThe instruction is essential beacuse hallucination here is compliance and fairnes risk, if the summariser envents details or fabricated \nclaim that is not stated in the letter it could influence a lending decision which might later poses dan

In [ ]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

import json
import pandas as pd

EXTRACT_SYSTEM_PROMPT = """You are a data extraction assistant for a microfinance loan officer.
Extract structured data from loan application letters and return ONLY a valid JSON object
no explanation, no markdown formatting, no code fences, just the raw JSON.

The JSON object must have EXACTLY these keys:
- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

If a field is not explicitly stated in the letter, use null. Do not guess or infer.

Example:
Letter:
"Dear Sir, my name is Adama Baba, a student at Ashesi University.
I would appreciate a loan of about GHS 2,500 to expand on my littel business I am doing in school,
which is selling of basic student accesories, of late the demands for my goods has been very high and
I need money to restock with high-quality products to expand my business.
I would be able to pay an amount of GHS 1,000 each month from my monthly stipens, making up to 3 months.

Output:
{"applicant_name": "Adama Baba", "amount_ghs": 2500, "purpose": "expand business", "monthly_profit_ghs": null, "has_collateral_or_guarantor": false, "repayment_months": 3}
"""


def build_extract_prompt(letter_text):
    return f"Letter:\n\"{letter_text}\"\n\nOutput:"



In [ ]:
def extract_fields(letter_text):
    prompt = build_extract_prompt(letter_text)
    raw = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM_PROMPT, temperature=0)

    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
        cleaned = cleaned.strip()

    try:
        result = json.loads(cleaned)
        return result
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON. Error: {e}")
        print(f"Raw output was:\n{raw}")
        return None

rows = []
for letter_id, text in LETTERS.items():
    result = extract_fields(text)
    if result is not None:
        result["letter_id"] = letter_id
        rows.append(result)
    else:
        rows.append({"letter_id": letter_id, "applicant_name": None, "amount_ghs": None,
                     "purpose": None, "monthly_profit_ghs": None,
                     "has_collateral_or_guarantor": None, "repayment_months": None})

df = pd.DataFrame(rows)
df = df[["letter_id", "applicant_name", "amount_ghs", "purpose",
         "monthly_profit_ghs", "has_collateral_or_guarantor", "repayment_months"]]
df

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair trotro engine and settle debts,NaN,False,NaN
2,L003,Efua Darko,15000,purchase industrial sewing machines and fabric...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,poultry farm,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy bulk order of yarn,NaN,True,16.0
5,L006,Kofi,50000,"start car washing business, provision shop, an...",NaN,False,12.0


In [4]:
"""Student Reasoning — Structured extraction 1. Why must the few-shot example NOT come from the six letters you are processing?
If the example come from the six letters the test set will be contaminated because I will basically be showing the model the answer and
later if the model later evaluate that same example and gets it right the accuracy score might shoot up and you think the model is
actualy performing well but it might not be able to generalize well on unseen data.

2. Why "use null, do not guess" — what did the model do without that instruction?
Without the instruction the model tries to be helpfull by filling every field even when the letter doesn't state a value.

3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?
Extraction is a task with one correct answer per letter, there is no room for creative variation, and the temperature = 0 removes
randomness so the letter always yields the same structured data, while creative tasks have many equally valid answers, and only
requires temperature that encourages exploring different options"""

'Student Reasoning — Structured extraction 1. Why must the few-shot example NOT come from the six letters you are processing? \nIf the example come from the six letters the test set will be contaminated because I will basically be showing the model the answer and \nlater if the model later evaluate that same example and gets it right the accuracy score might shoot up and you think the model is \nactualy performing well but it might not be able to generalize well on unseen data.\n\n2. Why "use null, do not guess" — what did the model do without that instruction? \nWithout the instruction the model tries to be helpfull by filling every field even when the letter doesn\'t state a value.\n\n3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?\nExtraction is a task with one correct answer per letter, there is no room for creative variation, and the temperature = 0 removes \nrandomness so the letter always yields the same structured data, while creativ

In [ ]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

BRIEF_SYSTEM_PROMPT = """You are a decision-support assistant for a microfinance loan officer.
You do NOT make lending decisions you help the human officer prepare to make one.
Given a loan application letter and its extracted structured data, produce a brief with
exactly these four sections:
1. Strengths (bullet points, grounded only in facts stated in the letter or the extracted data).
2. Risks / Red Flags, bullet points identifying anything concerning: vague plans, no
   collateral, inconsistent numbers, unrealistic assumptions, urgency without justification, etc.
3. Missing Information, bullet points listing what the officer should ask for before deciding.
4. Suggested Next Step ONE of: "invite for interview", "request documents", or
   "flag for senior review". NEVER say "approve" or "reject" the final decision is made
   by the human loan officer, not you.
Be factual and specific. Do not invent details not present in the letter or extracted data."""

def build_brief_prompt(letter_text, extracted_json):
    return (
        f"Loan application letter:\n\"{letter_text}\"\n\n"
        f"Extracted data:\n{json.dumps(extracted_json, indent=2)}\n\n"
        f"Produce the four-section brief."
    )

def generate_brief(letter_text, extracted_json):
    prompt = build_brief_prompt(letter_text, extracted_json)
    return ask_llm(prompt, system_prompt=BRIEF_SYSTEM_PROMPT, temperature=0)


briefs = {}
for letter_id, text in LETTERS.items():
    extracted = extract_fields(text)  # reuse Stage 2 extraction
    briefs[letter_id] = generate_brief(text, extracted)

# Print briefs for L001, L002, L006
for letter_id in ["L001", "L002", "L006"]:
    print(f"BRIEF {letter_id}")
    print(briefs[letter_id])
    print()

BRIEF L001
## Step 1: Strengths
The applicant, Akosua Mensah, has the following strengths:
* 12 years of experience selling provisions at Makola Market, indicating a stable business history.
* A consistent monthly profit of GHS 900, demonstrating a viable business.
* Savings of GHS 2,500 with the susu scheme over two years, showing financial discipline and a track record of regular contributions.
* A guarantor, her sister, a teacher, which provides an added layer of security for the loan.
* A clear plan for repayment, with a proposed monthly repayment amount of GHS 450 over 20 months.

## Step 2: Risks / Red Flags
The following risks and red flags are identified:
* The expansion into frozen foods may require additional skills or knowledge, and there is no indication that the applicant has experience in this area.
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit, and there is a risk that the business may not generate sufficient income to meet the 

In [5]:
"""Student Reasoning — Decision support 1. Compare the briefs for L003 (strong application) and L006 (weak application).
Did the system identify the right strengths and red flags in each?


2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and one ethical reason."""

'Student Reasoning — Decision support 1. Compare the briefs for L003 (strong application) and L006 (weak application).\nDid the system identify the right strengths and red flags in each? \n\n\n2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and one ethical reason.'

In [ ]:
"""commit hash: cc91df73a8083a5cabbac4c948ed184a7dd528d2"""

'commit hash: cc91df73a8083a5cabbac4c948ed184a7dd528d2'

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

import pandas as pd

fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
          "has_collateral_or_guarantor", "repayment_months"]
gold_ids = ["L001", "L003", "L006"]

def values_match(field, extracted_val, gold_val):
    if extracted_val is None and gold_val is None:
        return True
    if extracted_val is None or gold_val is None:
        return False
    if field == "applicant_name" or field == "purpose":
        # case-insensitive string comparison
        return str(extracted_val).strip().lower() == str(gold_val).strip().lower()
    else:
        # exact match for numbers and booleans
        return extracted_val == gold_val

# Build comparison table: rows = fields, columns = L001 / L003 / L006 / accuracy
comparison_rows = []
for field in fields:
    row = {"field": field}
    match_count = 0
    for letter_id in gold_ids:
        extracted_row = df[df["letter_id"] == letter_id].iloc[0]
        extracted_val = extracted_row[field]
        gold_val = GOLD[letter_id][field]
        is_match = values_match(field, extracted_val, gold_val)
        row[letter_id] = "thick" if is_match else f" cross (got: {extracted_val}, gold: {gold_val})"
        if is_match:
            match_count += 1
    row["accuracy"] = f"{match_count}/{len(gold_ids)} ({match_count/len(gold_ids):.0%})"
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,field,L001,L003,L006,accuracy
0,applicant_name,thick,thick,thick,3/3 (100%)
1,amount_ghs,thick,thick,thick,3/3 (100%)
2,purpose,cross (got: buy a deep freezer and expand int...,cross (got: purchase industrial sewing machin...,"cross (got: start car washing business, provi...",0/3 (0%)
3,monthly_profit_ghs,thick,thick,"cross (got: nan, gold: None)",2/3 (67%)
4,has_collateral_or_guarantor,thick,thick,thick,3/3 (100%)
5,repayment_months,thick,thick,thick,3/3 (100%)


In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

import json

L004_text = LETTERS["L004"]

def extract_fields_at_temp(letter_text, temperature):
    prompt = build_extract_prompt(letter_text)
    raw = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM_PROMPT, temperature=temperature)

    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
        cleaned = cleaned.strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON at temp={temperature}. Error: {e}")
        print(f"Raw output was:\n{raw}")
        return None
temp_0_results = []
print("Temperature = 0.0")
for i in range(5):
    result = extract_fields_at_temp(L004_text, temperature=0)
    temp_0_results.append(result)
    print(f"Run {i+1}: {result}")

temp_1_results = []
print("\nTemperature = 1.0 ")
for i in range(5):
    result = extract_fields_at_temp(L004_text, temperature=1.0)
    temp_1_results.append(result)
    print(f"Run {i+1}: {result}")

def analyze_consistency(results, label):
    valid = [r for r in results if r is not None]
    n_valid = len(valid)

    # Serialize each valid result with sorted keys so identical dicts produce identical strings
    serialized = [json.dumps(r, sort_keys=True) for r in valid]
    unique_strings = set(serialized)
    n_identical = max(serialized.count(s) for s in unique_strings) if serialized else 0

    print(f"{label}")
    print(f"Valid JSON: {n_valid}/5")
    print(f"Unique outputs: {len(unique_strings)}")
    print(f"Largest group of identical outputs: {n_identical}/5")
    print()

analyze_consistency(temp_0_results, "Temperature = 0.0")
analyze_consistency(temp_1_results, "Temperature = 1.0")


Temperature = 0.0
Run 1: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 2: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 3: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 4: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 5: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}

Temperature = 1.0 
Run 1: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'rebuild po

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

test1_question_prompt = (
    "Here is a loan application letter:\n\n"
    f"{LETTERS['L001']}\n\n"
    "Question: What is the applicant's credit score?"
)

test1_system_prompt = (
    "You are an assistant to a microfinance loan officer. "
    "Answer questions about loan application letters factually. "
    "If the information requested is not stated in the letter, say so explicitly. "
    "Do not invent or guess any detail not present in the letter."
)

test1_output = ask_llm(test1_question_prompt, system_prompt=test1_system_prompt, temperature=0)


print("ADVERSARIAL TEST 1 Asking for absent detail (credit score)")
print(test1_output)

ADVERSARIAL TEST 1 Asking for absent detail (credit score)
The applicant's credit score is not stated in the letter. The letter mentions that the applicant has never missed a contribution to the susu scheme, which suggests a good repayment history, but it does not provide a specific credit score.


In [ ]:
weather_report = (
    "Today's forecast for Accra: partly cloudy with a high of 31°C and a low of 24°C. "
    "Humidity around 78%. Light winds from the southwest. Chance of rain in the "
    "afternoon, clearing by evening. Tomorrow expected to be similar with slightly "
    "cooler temperatures."
)

test2_output = extract_fields(weather_report)

print("ADVERSARIAL TEST 2 — Extracting from an irrelevant text (weather report)")
print(test2_output)



ADVERSARIAL TEST 2 — Extracting from an irrelevant text (weather report)
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


In [6]:
"""Student Reasoning — Evaluation results 1. Report your extraction accuracy. Which field was hardest for the model and why?
The hardest field was purpose and it was because of measurement limitations of exact string matching against free text.

2. What did the reliability experiment show about temperature and production systems?
It showed that at creative temperature structured numeric or boolean fields stayed stable while the one field asking for natural language
paraphrase was the only one to drift.

3. Did your system hallucinate under probing? If yes, how could the prompt (or the system design around it) reduce the risk?
The system did not halluciate."""

'Student Reasoning — Evaluation results 1. Report your extraction accuracy. Which field was hardest for the model and why? \nThe hardest field was purpose and it was because of measurement limitations of exact string matching against free text.\n\n2. What did the reliability experiment show about temperature and production systems? \nIt showed that at creative temperature structured numeric or boolean fields stayed stable while the one field asking for natural language \nparaphrase was the only one to drift. \n\n3. Did your system hallucinate under probing? If yes, how could the prompt (or the system design around it) reduce the risk?\nThe system did not halluciate.'

In [7]:
"""Student Reasoning — Appropriateness
1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions with your system, who could be unfairly harmed,
and how?  Consider applicants who write poorly in English but run solid businesses.
Applicants who write poor english are obviously those who will be harmed, because the system summarises and infer risk partly from
how an applicant wirtes.

2. Loan letters contain personal data. What are the implications of sending them to a third-party API in another country?
What would you check before deploying this at a real Ghanaian microfinance institution?
The implications of sedning them to a third party API means the data leaves leaves Ghana and is now subject to the data protection laws
of another country. Before deploying the model in Ghan you have to check whether it complies with the Ghana's Data Protection Act(2012)
and any Bank of Ghana requirements for handling customer financial data.

3. Name TWO concrete safeguards you would build around this system in production (think: human review points, logging, appeal processes, monitoring).
Mandatory human review before any adverse action, logging and periodic bias audits."""

"Student Reasoning — Appropriateness \n1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions with your system, who could be unfairly harmed, \nand how?  Consider applicants who write poorly in English but run solid businesses. \nApplicants who write poor english are obviously those who will be harmed, because the system summarises and infer risk partly from \nhow an applicant wirtes.\n\n2. Loan letters contain personal data. What are the implications of sending them to a third-party API in another country? \nWhat would you check before deploying this at a real Ghanaian microfinance institution? \nThe implications of sedning them to a third party API means the data leaves leaves Ghana and is now subject to the data protection laws \nof another country. Before deploying the model in Ghan you have to check whether it complies with the Ghana's Data Protection Act(2012) \nand any Bank of Ghana requirements for handling customer financial data.\n\n3. Name T

In [11]:
"""Prompting as engineering: How is iterating on a prompt similar to and different from iterating on the model hyperparameters you tuned in Lab 3?
Similar: Both iterating on a promt and iterating on the model hyperparameters are empirical processes, you change one variable, run it,
observe the output, and adjust based on results rather than getting it right analytically on the first try.
Both require systematic testing
Different: Hyperparameter tuning in Lab 3 optimizes a numeric objective (loss, accuracy) that you can measure precisely and automatically.
Prompt engineering optimizes output quality that often require human judgment to evaluate, and small wording changes can produce
disproportionately large behavioral shifts.

Trust: After your Section 4 evaluation, would you trust this system to run unattended? What single evaluation result most influenced your answer?
No I cannot trust the system to run unattended to because of the imperfectfield level extraction accuracy especially on purpose,
some output variation at higer temperature.

Cost and scale: Estimate (from your response.usage numbers) the tokens needed to process 1,000 applications per month. What does that imply for provider choice?
57 tokens * 3 stages * 1, 000 applications per month = 171, 000 tokens per month. Which means this task's token volume is large enough that free tier development
setups will not survive production, but small enough per call that a cheap, fast, open model provider is still the right category of choice the constraint.

Looking back at the course: You have now used classical ML (Lab 2), trained neural networks (Lab 3), and used a foundation model via
API (Lab 4). For a task like this one, why does calling an API beat training your own model — and when would it not?
Calling an API beats training your own model because you get access to a model trained on internet scale data with strong general
language understanding, with zero training cost, zero infrastructure to maintain, and results in hours instead of weeks.
For a task like structured extraction and summarization from natural language which requires broad language competence,
not narrow domain specific pattern matching a general purpose LLM already has what is needed."""

"Prompting as engineering: How is iterating on a prompt similar to and different from iterating on the model hyperparameters you tuned in Lab 3?\nSimilar: Both iterating on a promt and iterating on the model hyperparameters are empirical processes, you change one variable, run it, \nobserve the output, and adjust based on results rather than getting it right analytically on the first try. \nBoth require systematic testing\nDifferent: Hyperparameter tuning in Lab 3 optimizes a numeric objective (loss, accuracy) that you can measure precisely and automatically. \nPrompt engineering optimizes output quality that often require human judgment to evaluate, and small wording changes can produce \ndisproportionately large behavioral shifts.\n\nTrust: After your Section 4 evaluation, would you trust this system to run unattended? What single evaluation result most influenced your answer?\nNo I cannot trust the system to run unattended to because of the imperfectfield level extraction accuracy e